# Sales Churn Prediction Model

This notebook loads customer sales data from Snowflake, engineers features, and trains a classification model to predict customer churn.

**Dataset:** Customer transactions, demographics, and engagement metrics

**Target:** `IS_CHURNED` (1 = churned, 0 = active)

In [ ]:
import pandas as pd
import numpy as np
import os

from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
%%sql -r dataframe_1

use database analytics;
use schema analytics.sales;

SELECT
    c.CUSTOMER_ID,
    c.AGE,
    c.REGION,
    c.ACCOUNT_LENGTH_DAYS,
    c.CONTRACT_TYPE,
    s.TOTAL_PURCHASES,
    s.AVG_ORDER_VALUE,
    s.DAYS_SINCE_LAST_PURCHASE
    s.TOTAL_RETURNS,
    e.SUPPORT_TICKETS,
    e.AVG_RESPONSE_SCORE,
    e.MONTHLY_LOGINS,
    c.IS_CHURNED
FROM ANALYTICS.SALES.CUSTOMERS c
JOIN ANALYTICS.SALES.PURCHASE_SUMMARY s ON c.CUSTOMER_ID = s.CUSTOMER_ID
JOIN ANALYTICS.SALES.ENGAGEMENT e ON c.CUSTOMER_ID = e.CUSTOMER_ID


In [ ]:
df = dataframe_1
print(f"Loaded {len(df)} rows with {len(df.columns)} columns")
df.head()

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nChurn distribution:")
print(df["IS_CHURNED"].value_counts(normalize=True))
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nBasic stats:")
df.describe()

In [ ]:
# Feature Engineering
df["PURCHASE_FREQUENCY"] = df["TOTAL_ORDERS"] / (df["ACCOUNT_LENGTH_DAYS"] / 30)

df["RETURN_RATE"] = df["TOTAL_RETURNS"] / df["TOTAL_PURCHASES"].replace(0, np.nan)
df["RETURN_RATE"] = df["RETURN_RATE"].fillna(0)

df["ENGAGEMENT_SCORE"] = (
    df["MONTHLY_LOGINS"] * 0.4 +
    df["AVG_RESPONSE_SCORE"] * 0.3 +
    (1 / (df["DAYS_SINCE_LAST_PURCHASE"] + 1)) * 0.3
)

df["CONTRACT_TYPE_ENCODED"] = df["CONTRACT_TYPE"].map({
    "Monthly": 0, "Annual": 1, "Two-Year": 2
})

df["REGION_ENCODED"] = pd.Categorical(df["REGION"]).codes

print(f"Features engineered. Shape: {df.shape}")
df[["PURCHASE_FREQUENCY", "RETURN_RATE", "ENGAGEMENT_SCORE"]].describe()

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "AGE", "ACCOUNT_LENGTH_DAYS", "TOTAL_PURCHASES", "AVG_ORDER_VALUE",
    "DAYS_SINCE_LAST_PURCHASE", "TOTAL_RETURNS", "SUPPORT_TICKETS",
    "AVG_RESPONSE_SCORE", "MONTHLY_LOGINS", "PURCHASE_FREQUENCY",
    "RETURN_RATE", "ENGAGEMENT_SCORE", "CONTRACT_TYPE_ENCODED",
    "REGION_ENCODED", "IS_CHURNED"
]

X = df[feature_cols]
y = df["IS_CHURNED"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

In [ ]:
from sklearn.linear_model import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model.fit(X_train_scaled, y_train)
print("Model trained successfully")

train_score = model.score(X_train_scaled, y_train)
test_score = model.score(X_test_scaled, y_test)
print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Classification Report:")
print(classification_report(y_pred, y_test))

print("\nConfusion Matrix:")
print(confusion_matrix(y_pred, y_test))

print(f"\nROC AUC: {roc_auc_score(y_pred, y_pred_proba):.4f}")